In [4]:
# 1. Carga das bibliotecas

# Manipulação e análise dos dados tabulares
import pandas as pd
import numpy as np

# Gráficos e visualizações estatísticas
import matplotlib.pyplot as plt
import seaborn as sns

# Divisão dos dados em treino e teste
from sklearn.model_selection import train_test_split

# Algoritmo de Machine Learning: Random Forest (Florestas Aleatórias)
from sklearn.ensemble import RandomForestClassifier

# Métricas para medir o quão bom ficou o modelo
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


In [5]:
# Carregamos o train.csv (que tem as respostas) e o test.csv (para ver a diferença)
df_treino_kaggle = pd.read_csv('../data/train.csv')
df_teste_kaggle = pd.read_csv('../data/test.csv')

print(f"Colunas do train.csv: {len(df_treino_kaggle.columns)}")
print(f"Colunas do test.csv:  {len(df_teste_kaggle.columns)} (não tem a coluna 'Survived')")

df_treino_kaggle.head(3)

Colunas do train.csv: 12
Colunas do test.csv:  11 (não tem a coluna 'Survived')


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


In [6]:
# 1. Descartar identificadores
colunas_remover = ['PassengerId', 'Name', 'Ticket', 'Cabin']
df_clean = df_treino_kaggle.drop(columns=colunas_remover)

# 2. Preencher nulos (imputação)
df_clean['Age'] = df_clean['Age'].fillna(df_clean['Age'].median())
df_clean['Embarked'] = df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0])
df_clean['Fare'] = df_clean['Fare'].fillna(df_clean['Fare'].median())

# 3. Converter texto para binário
df_clean = pd.get_dummies(df_clean, columns=['Sex', 'Embarked'], drop_first=True)

df_clean.head(3)

,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S
0,0,3,22.0,1,0,7.2500,True,False,True
1,1,1,38.0,1,0,71.2833,False,False,False
2,1,3,26.0,0,0,7.9250,False,False,True


In [7]:
# Separar features (X) e rótulo (y)
X = df_clean.drop('Survived', axis=1)
y = df_clean['Survived']

# 80% para treino e 20% para teste local
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"Linhas para treinar: {X_treino.shape[0]}")
print(f"Linhas para testar:  {X_teste.shape[0]}")

Linhas para treinar: 712
Linhas para testar:  179


In [8]:
modelo = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
modelo.fit(X_treino, y_treino)

print("Modelo treinado com sucesso!")

Modelo treinado com sucesso!


In [9]:
# Previsões na amostra de teste local
y_pred = modelo.predict(X_teste)

# Acurácia
acuracia = accuracy_score(y_teste, y_pred)
print(f"Acurácia: {acuracia * 100:.2f}%\n")

# Relatório
print("--- Relatório de Classificação ---")
print(classification_report(y_teste, y_pred, target_names=['Não Sobreviveu', 'Sobreviveu']))

# Matriz de Confusão
print("--- Matriz de Confusão ---")
print(confusion_matrix(y_teste, y_pred))

Acurácia: 78.21%

--- Relatório de Classificação ---
                precision    recall  f1-score   support

Não Sobreviveu       0.78      0.90      0.84       110
    Sobreviveu       0.79      0.59      0.68        69

      accuracy                           0.78       179
     macro avg       0.78      0.75      0.76       179
  weighted avg       0.78      0.78      0.77       179

--- Matriz de Confusão ---
[[99 11]
 [28 41]]


In [10]:
# Preparar test.csv
ids = df_teste_kaggle['PassengerId']
X_submissao = df_teste_kaggle.drop(columns=colunas_remover)
X_submissao['Age'] = X_submissao['Age'].fillna(df_treino_kaggle['Age'].median())
X_submissao['Fare'] = X_submissao['Fare'].fillna(df_treino_kaggle['Fare'].median())
X_submissao = pd.get_dummies(X_submissao, columns=['Sex', 'Embarked'], drop_first=True)

# Gerar arquivo para envio ao Kaggle
previsoes_kaggle = modelo.predict(X_submissao)
submissao = pd.DataFrame({'PassengerId': ids, 'Survived': previsoes_kaggle})
submissao.to_csv('../data/minha_submissao.csv', index=False)

print("Arquivo 'minha_submissao.csv' gerado na pasta data/!")

Arquivo 'minha_submissao.csv' gerado na pasta data/!


De todas as previsões feitas pelo modelo, ele acertou 78,21% das vezes (140 acertos de 179 passageiros). Para um primeiro modelo básico no Titanic, esse é um resultado inicial padrão e coerente.  2. Matriz de Confusão
Ela mostra o cruzamento exato entre o que realmente aconteceu e o que o modelo previu:  Modelo previu: Não SobreviveuModelo previu: SobreviveuReal: Não Sobreviveu99 (Acertou)  11 (Errou: disse que sobreviveu)  Real: Sobreviveu28 (Errou: disse que morreu)  41 (Acertou)  O modelo acertou $99 + 41 = 140$ passageiros.  O modelo errou $11 + 28 = 39$ passageiros.  3. Relatório de Classificação
Ele detalha o desempenho separando por classe:  Support: Quantidade real de pessoas na amostra de teste. Eram 110 não sobreviventes e 69 sobreviventes.  Precision (Precisão):Quando o modelo previu que alguém Sobreviveu, ele estava certo em 79% dos casos ($41 / (41 + 11)$).  Quando previu que alguém Não Sobreviveu, estava certo em 78%.  Recall (Revocação / Sensibilidade):Para Não Sobreviveu: 0.90 (90%). Dos 110 que realmente morreram, ele identificou 99.  Para Sobreviveu: 0.59 (59%). Aqui está a maior fragilidade do modelo: dos 69 sobreviventes reais, ele só conseguiu capturar 41, deixando 28 passarem batido.  F1-score: A média harmônica entre Precision e Recall. Serve para dar uma nota equilibrada: 0.84 para mortes e 0.68 para sobrevivências.  O modelo é muito bom em identificar quem não sobreviveu (90% de captura), mas é mais conservador ao prever quem sobreviveu, deixando escapar 41% dos sobreviventes reais. Para melhorar essa taxa, o próximo passo comum em Machine Learning é criar novas variáveis (como o tamanho da família ou o título extraído do nome).  